# PC-FNO3D national — règle GFS v5.1, partie B1 (Colab A100) : affinage C1, étapes 1 + 3 (validation 2021)

Règle `configs/national/regle_national_gfs_v5_1.json` (sha256 `570713841039…`), approuvée par Ines le 22/11/2026 (soir) et vérifiée par le carnet ; elle remplace `regle_national_gfs_v5` (jamais exécutée). Code : dépôt public au commit `b3ac84a93e34` (le carnet refuse de tourner tant qu'il n'est pas renseigné). Sorties : `MyDrive/pcfno3d_national/gfs_v5_regle/` et `runs/national_1km_v5_gfs` (dossiers neufs, rien n'est supprimé ni écrasé). Choix sur la validation 2021 seulement ; 2022 n'est pas relu ; l'échantillon 2026 (jours impairs, 00/06/12/18 UTC, ~608 instants) n'est lu qu'une fois, et seulement si une variante passe les trois cases sur 2021 puis si les parties vent de l'entrée de production sont identiques à celles du cache. GFS et ERA5 ne sont jamais appelés mesures (seuls les METAR le sont). Aucun bruit ajouté. Plafond de la règle : 90 unités Colab (A + B1 + B2) ; le coût affiché est une estimation. La session est libérée à la fin.

## 0. Commit figé, budget, GPU (A100 obligatoire)

In [ ]:
# Commit FIGE du code (depot public). Tant qu'il n'est pas renseigne, rien ne tourne : session liberee.
import re as _re
COMMIT_FIGE = "b3ac84a93e34c4bef6a26a07cc95214973fce083"
if not _re.fullmatch(r"[0-9a-f]{40}", COMMIT_FIGE):
    from google.colab import runtime
    print("commit du code non renseigne (scripts/make_national_gfs_v5_nb.py, COMMIT) : rien n'est lance, "
          "session liberee")
    runtime.unassign()
    raise SystemExit("commit non fige")
print("commit fige :", COMMIT_FIGE)

In [ ]:
# BUDGET (regle v5.1) : plafond 90 unites Colab pour toute la regle (A + B1 + B2). Renseigner les unites DEJA
# consommees par la regle (0 avant la partie A) en lisant le solde Colab. Cout de cette partie : ESTIMATION.
UNITES_DEJA_UTILISEES = None
_mat, _taux, _hmin, _hmax = "A100", 11.8, 2.0, 3.0
_bas, _haut = _taux * _hmin, _taux * _hmax
print(f"cout ESTIME de la partie B1 (A100, ~11.8 unites/h, estimation) : {_hmin:g}-{_hmax:g} h, "
      f"soit ~{_bas:.0f}-{_haut:.0f} unites ; plafond de la regle : 90 unites (A + B1 + B2)")
if UNITES_DEJA_UTILISEES is None or float(UNITES_DEJA_UTILISEES) + _haut > 90:
    from google.colab import runtime
    print("budget : UNITES_DEJA_UTILISEES non renseigne, ou plafond risque d'etre depasse : rien n'est lance, "
          "demander a Ines (regle v5.1, « budget ») ; session liberee")
    runtime.unassign()
    raise SystemExit("budget non verifie")
print(f"budget : {float(UNITES_DEJA_UTILISEES):g} deja utilisees + {_haut:.0f} au plus <= 90")

In [ ]:
# Le national (453 M parametres, grille 16 x 828 x 388) ne tient pas sur une L4 de 22 Go, meme avec le
# recalcul des activations (3 echecs le 11/11, dont un avec le carnet corrige : 20,8 Go + 2,45 Go demandes).
# Si Colab n'attribue pas un GPU d'au moins 30 Go (A100), la session est liberee AVANT le montage du Drive.
import subprocess as _sp
_g = _sp.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
             capture_output=True, text=True)
_nom, _mo = ([s.strip() for s in _g.stdout.strip().splitlines()[0].split(",")] if _g.returncode == 0
             and _g.stdout.strip() else ("aucun GPU", "0"))
print("GPU attribue :", _nom, f"{float(_mo) / 1024:.0f} Go")
if float(_mo) < 30 * 1024:
    from google.colab import runtime
    print("GPU trop petit pour le national : session liberee, rien n'a ete lance. Relancer plus tard (A100).")
    runtime.unassign()
    raise SystemExit("GPU de moins de 30 Go")

## 1. Drive, code au commit figé, données

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")   # code et poids du Grand Tunis
NAT = Path("/content/drive/MyDrive/pcfno3d_national")
NAT.mkdir(parents=True, exist_ok=True)
JOURNAUX = NAT / "journaux"
JOURNAUX.mkdir(exist_ok=True)
libre = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"place libre sur le Drive : {libre:.0f} Go (le jeu national en demande ~50)")

CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille nationale, grille_reelle.json, forcage ERA5 (une archive sur le Drive)
DON = Path("/content/donnees")
G = DON / "national"
if not (G / "forcage_era5.csv").exists():
    with zipfile.ZipFile(NAT / "national_donnees.zip") as zf:
        zf.extractall(DON)
print(sorted(p.name for p in G.iterdir()))

# code du depot public au commit fige (remplace l'archive du Drive, plus ancienne)
COMMIT = "b3ac84a93e34c4bef6a26a07cc95214973fce083"
os.chdir("/content")    # le noyau etait dans /content/PFE : le supprimer laisserait pip dans un dossier efface
subprocess.run(f"rm -rf /content/PFE && git init -q /content/PFE && cd /content/PFE && git remote add origin "
               f"https://github.com/ines-rouatbi/pcfno3d.git && git fetch -q --depth 1 origin {COMMIT} "
               f"&& git checkout -q FETCH_HEAD", shell=True, check=True)
os.chdir("/content/PFE")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "/content/PFE"],
                   capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install -e /content/PFE a echoue (sortie ci-dessus)")
import importlib
importlib.invalidate_caches()
import hashlib
for f, h in (("regle_v4_national.json", "60671e843ee648ee2422168a7659a53dfe8aa414be7b1c7c6f54eeec622a43f8"), ("regle_national_gfs_v4.json", "4ca1f2a0c34856d6840d620cd9457534c37cb316a59080400e2853e2f0d7d721"), ("regle_national_gfs_v5_1.json", "570713841039a5fda8e4d8727a047a4c6a3171a9c212b5402cdb2b7356be9cdd")):
    assert hashlib.sha256((CODE / "configs" / "national" / f).read_bytes()).hexdigest() == h, f
print("code", COMMIT, "| regles v4, gfs_v4 et gfs_v5_1 verifiees")

GV5 = NAT / "gfs_v5_regle"                 # dossier NEUF de la regle (rien d'autre n'y ecrit)
GV5.mkdir(exist_ok=True)
ECH = (0, 24, 48)
F = {e: GV5 / f"forcage_gfs_f{e}.csv" for e in ECH}                 # entrainement 2023-2025 + validation 2021
TN = {e: GV5 / f"gfs_f{e}_tn" for e in ECH}                         # echeances telechargees (boite nationale)
F26 = {e: GV5 / "jugement_2026" / f"forcage_gfs_2026_f{e}.csv" for e in ECH}
TN26 = {e: GV5 / f"gfs_2026_f{e}_tn" for e in ECH}
# archives deja telechargees, LUES SEULEMENT (rien n'y est ecrit) : OS7 du Grand Tunis, OS7 national s'il existe
LECTURE = {e: [d for d in (GT / "os7" / f"gfs_f{e}_instants", NAT / "os7" / f"gfs_f{e}_tn") if d.exists()] for e in ECH}
VAL = GV5 / "validation_2021"              # etapes 1 + 3
RES = GV5 / "resultat"                     # etape 4 (controle, puis jugement 2026)
METAR26 = GV5 / "metar_2026"               # METAR 2026 (archive IEM), telecharges seulement a l'etape 4
RCP = NAT / "runs" / "national_1km_v4_Cp_s44"          # C' s44 tel que livre (lu seulement)
R5 = NAT / "runs" / "national_1km_v5_gfs"              # C1 : run NEUF
SHA_V5_1 = "570713841039a5fda8e4d8727a047a4c6a3171a9c212b5402cdb2b7356be9cdd"
# echantillon 2026 de la regle v5.1 : jours de l'annee impairs, heures de cycle (analyse toujours f000)
PERIODE_2026 = "--periode 2026-01-01T00:00,2026-10-31T18:00 --jours-impairs --heures 0,6,12,18"
print("accord :", "Ines, 22/11/2026 (soir) : regle_national_gfs_v5_1 approuvee et hachee (570713841039...)")

## 2. Mémoire du GPU (recalcul des activations si besoin)

In [ ]:
# Essai de memoire dans un PROCESSUS SEPARE : le noyau du carnet ne touche jamais au GPU (le 11/11, l'essai fait
# dans le noyau y gardait 3,3 Go et l'entrainement sur L4 22 Go a manque de memoire). Allocateur a segments
# extensibles (moins de fragmentation), herite par les commandes lancees ensuite. Aucun effet sur les calculs.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
ESSAI = r"""
import json, sys, torch
from pcfno3d.config import Config
from pcfno3d.geometry import Geometry
from pcfno3d.model import build_model
recalcul = sys.argv[1] == "1"
cfg = Config.load("configs/national_1km_colab.yaml")
cfg.model.recalcul_activations = recalcul
geom = Geometry(cfg, device="cuda")
m = build_model(cfg, geom.c_in).cuda().train()
x = torch.randn(cfg.train.batch_size, geom.c_in, *geom.shape, device="cuda")
torch.cuda.reset_peak_memory_stats()
try:
    with torch.autocast("cuda", dtype=torch.bfloat16):
        m(x).float().pow(2).mean().backward()
    pic = torch.cuda.max_memory_allocated() / 1e9
except torch.cuda.OutOfMemoryError:
    pic = None
p = torch.cuda.get_device_properties(0)
print("ESSAI", json.dumps({"gpu": p.name, "total": p.total_memory / 1e9, "pic": pic}))
"""
Path("/content/essai_memoire.py").write_text(ESSAI)
def essai(recalcul: bool) -> dict:
    r = subprocess.run([sys.executable, "/content/essai_memoire.py", "1" if recalcul else "0"], cwd="/content/PFE",
                       capture_output=True, text=True)
    ligne = next((l for l in r.stdout.splitlines() if l.startswith("ESSAI ")), None)
    if ligne is None:
        print(r.stdout[-2000:], r.stderr[-2000:])
        return {"gpu": "?", "total": float("nan"), "pic": None}
    return json.loads(ligne[6:])
e = essai(False)
total = e["total"]
RECALCUL = e["pic"] is None or e["pic"] > 0.85 * total
if RECALCUL:
    e = essai(True)
print(e["gpu"], f"{total:.0f} Go | pic de l'essai {e['pic'] if e['pic'] is None else round(e['pic'], 1)} Go "
      f"| recalcul des activations : {RECALCUL}")
if total < 30:
    print("ATTENTION : GPU de moins de 30 Go (L4 ?). Le national a manque de memoire sur L4 le 11/11 avec l'ancien "
          "carnet ; cet essai ne compte ni l'optimiseur ni l'EMA. Preferer une A100.")
OPTS = f"--set model.recalcul_activations={str(RECALCUL).lower()}"

## 3. Cache v3 complet (entraînement de C1)

In [ ]:
LOC = Path("/content/national")
CACHE, JEU = LOC / "cache", LOC / "jeu"
for k in (1, 2, 3):
    if not (NAT / f"cache_v3_partie_{k}.zip").exists():
        raise FileNotFoundError(f"cache_v3_partie_{k}.zip absent : finir le carnet de generation")
if not (CACHE / "index.csv").exists():
    CACHE.mkdir(parents=True, exist_ok=True)
    for k in (1, 2, 3):
        with zipfile.ZipFile(NAT / f"cache_v3_partie_{k}.zip") as zf:
            zf.extractall(CACHE)
    with zipfile.ZipFile(NAT / "jeu_v3.zip") as zf:
        zf.extractall(JEU)
    import pandas as pd
    idx = pd.concat([pd.read_csv(CACHE / f"index_partie_{k}.csv") for k in (1, 2, 3)])
    idx.drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
import pandas as pd
print(pd.read_csv(CACHE / "index.csv")["split"].value_counts().to_string())
lancer("controle", "python -u -m pcfno3d -c national_1km_colab check")

## 4. C1, puis étapes 1 + 3 ; arrêt si la sélection dit ARRÊT ; fin de la session

In [ ]:
import shlex
def choisir_poids(src):
    """Le Drive garde parfois deux fichiers du meme nom (grand_tunis_v4_dso : deux best.pt de 1,69 Go,
    vus le 31/10) ; Colab peut n'en montrer qu'un, au hasard. On retient celui dont l'epoque est celle
    du test evalue (eval_pcfno3d/best_wcont, sinon best), meme s'il n'y en a qu'un : jamais un poids
    pris au hasard."""
    cands = sorted(src.glob("best*.pt"))
    if not cands:
        return src / "best.pt"
    attendu = next((json.loads(f.read_text()).get("epoque") for d in ("best_wcont", "best")
                    if (f := src / "eval_pcfno3d" / d / "test_summary.json").exists()), None)
    import torch
    epoques = {}
    for c in cands:
        ck = torch.load(str(c), map_location="cpu", mmap=True, weights_only=False)
        epoques[c] = ck.get("epoch", -2) + 1
        del ck
    print("poids candidats :", {c.name: e for c, e in epoques.items()}, "| epoque evaluee :", attendu)
    ok = [c for c in cands if epoques[c] == attendu] if attendu is not None else cands
    if not ok:
        raise RuntimeError(f"aucun poids de {src} n'a l'epoque evaluee ({attendu}) : verifie sur le Drive")
    return max(ok, key=lambda c: c.stat().st_mtime)          # a epoque egale : le plus recent

import yaml, shlex
import pandas as pd
def preparer_forcages_et_configs():
    """Forcages GFS (entrainement + validation) sur le disque local, configuration de C' s44 tel que livre."""
    for e in ECH:
        if not F[e].exists():
            raise FileNotFoundError(f"{F[e]} absent : lancer d'abord le carnet A (Colab CPU)")
    locf = G / "gfs_v5"                            # le chemin lu par configs/national_v5_gfs_mixte_colab.yaml
    locf.mkdir(exist_ok=True)
    for e in ECH:
        for suf in (".csv", ".npz"):
            if not (locf / F[e].with_suffix(suf).name).exists():
                shutil.copy2(F[e].with_suffix(suf), locf / F[e].with_suffix(suf).name)
    c = yaml.safe_load(open(CODE / "configs" / "national_1km_colab.yaml"))     # C' s44 tel que livre
    c["name"] = "national_v4_eval"
    c["paths"]["run_dir"], c["paths"]["local_dir"] = str(RCP), "/content/run_local_v4_sCp_s44"
    yaml.safe_dump(c, open(CODE / "configs" / "national_v4_eval.yaml", "w"), sort_keys=False)
    return locf

# PARTIE B1 (Colab A100) : C1, puis etapes 1 + 3 sur la validation 2021 ; rien de 2026 n'est lu ici
from google.colab import runtime
import traceback
try:
    LOCF = preparer_forcages_et_configs()
    INIT = choisir_poids(RCP)
    print("C' s44 :", INIT)
    # 1. C1 = national_1km_v5_gfs : run NEUF ; un dossier d'une autre origine n'est jamais touche
    marqueur = R5 / "regle_gfs_v5_1.json"
    if R5.exists() and any(R5.iterdir()) and not marqueur.exists():
        raise SystemExit(f"{R5} existe et ne vient pas de cette regle : rien n'est ecrase (verifier sur le Drive)")
    h = R5 / "history.json"
    fini = h.exists() and any(r.get("epoch") == 12 - 1 for r in json.loads(h.read_text()))
    commence = h.exists() or (R5 / "best.pt").exists()
    if commence and not fini and not Path("/content/run_local_national_1km_v5_gfs/last.pt").exists():
        cote = R5.with_name(R5.name + "_interrompu_" + time.strftime("%Y%m%d_%H%M%S"))   # mis de cote, JAMAIS supprime
        shutil.move(str(R5), str(cote))
        print("entrainement interrompu mis de cote :", cote)
    R5.mkdir(parents=True, exist_ok=True)
    if not marqueur.exists():
        marqueur.write_text(json.dumps({"regle": "regle_national_gfs_v5_1.json", "regle_sha256": SHA_V5_1,
                                        "depart": str(INIT), "commit": COMMIT}, indent=1))
    if not (R5 / "norm_stats.json").exists():      # normalisation de C' s44 (memes donnees d'entrainement)
        shutil.copy2(RCP / "norm_stats.json", R5 / "norm_stats.json")
    if not fini:
        lancer("entrainer_c1_gfs_v5", f"python -u -m pcfno3d -c national_v5_gfs_mixte_colab {OPTS} "
                                      f"--set train.init_from={shlex.quote(str(INIT))} train")
    INIT_C1 = choisir_poids(R5)
    print("C1 :", INIT_C1, "|", (R5 / "history.json").read_text()[-1500:])
    # 2. etapes 1 + 3 sur la validation 2021 (13 variantes x 3 cases, choix ; aucune lecture de 2022 ni de 2026)
    if not (VAL / "selection.json").exists():
        lancer("choisir_gfs_v5", f"python -u scripts/choisir_national_gfs_v5.py --cp national_v4_eval --c1 national_v5_gfs_mixte_colab "
                                 f"--gfs {LOCF} --obs {G / 'obs'} --sortie {VAL} "
                                 f"--checkpoint-cp {shlex.quote(str(INIT))} --checkpoint-c1 {shlex.quote(str(INIT_C1))}")
    sel = json.loads((VAL / "selection.json").read_text())
    print((VAL / "validation_gfs_v5.md").read_text())
    if sel["arret"]:
        print("ARRET (regle, etape 3) : aucune variante ne passe les trois cases sur 2021 ; 2026 n'est pas lu, "
              "le carnet B2 refusera de tourner")
    else:
        print("variante retenue :", sel["retenue"], "-> carnet B2 (L4), apres verification du budget")
except Exception:
    traceback.print_exc()
finally:
    runtime.unassign()